In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
B, T, C = 2, 8, 16
n_head = 4
head_dim = C // n_head

x = torch.randn(B, T, C)


In [ ]:
# Causal self-attention (one head shown, then multi-head)

# Single head
q = k = v = x
wei = q @ k.transpose(-2, -1) / math.sqrt(C)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
out = wei @ v
print("attention output:", out.shape)

# Multi-head (GPT-2 style)
c_attn = nn.Linear(C, 3 * C)
qkv = c_attn(x)
q, k, v = qkv.split(C, dim=2)
q = q.view(B, T, n_head, head_dim).transpose(1, 2)
k = k.view(B, T, n_head, head_dim).transpose(1, 2)
v = v.view(B, T, n_head, head_dim).transpose(1, 2)
att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
att = att.masked_fill(tril.view(1, 1, T, T) == 0, float("-inf"))
att = F.softmax(att, dim=-1)
y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
print("multi-head output:", y.shape)
